# Homework 3: Building an NDArray library

In this homework, you will build a simple backing library for the processing that underlies most deep learning systems: the n-dimensional array (a.k.a. the NDArray).  Up until now, you have largely been using numpy for this purpose, but this homework will walk you through developing what amounts to your own (albeit much more limited) variant of numpy, which will support both CPU and GPU backends.  What's more, unlike numpy (and even variants like PyTorch), you won't simply call out to existing highly-optimized variants of matrix multiplication or other manipulation code, but actually write your own versions that are reasonably competitive will the highly optimized code backing these standard libraries (by some measure, i.e., "only 2-3x slower" ... which is a whole lot better than naive code that can easily be 100x slower).  This class will ultimately be integrated into `needle`, but for this assignment you can _only_ focus on the ndarray module, as this will be the only subject of the tests.

**Note**: To avoid exhausting limited GPU resources in Colab, start by using CPU runtime for coding and testing non-GPU functions. Switch to GPU runtime when testing CUDA or GPU-accelerated code. This approach ensures efficient GPU usage and prevents running out of resources during critical tasks.

---

# 作业三：构建NDArray库

在本作业中，你将构建一个支撑大多数深度学习系统处理过程的基础库：n维数组（又称NDArray）。到目前为止，你主要使用numpy来实现这一目的，但本作业将引导你开发属于自己的（尽管功能更为有限）numpy变体，该库将同时支持CPU和GPU后端。更重要的是，与numpy（甚至像PyTorch这样的变体）不同，你不会简单地调用现有的高度优化的矩阵乘法或其他操作代码，而是实际编写自己版本的代码，使其在性能上与支撑这些标准库的高度优化代码具有合理竞争力（从某种衡量标准来看，即“仅慢2-3倍”……这比可能轻易慢100倍的朴素代码要好得多）。这个类最终将集成到`needle`中，但在本作业中，你应*仅*专注于ndarray模块，因为这是测试的唯一主题。

**注意**：为避免在Colab中耗尽有限的GPU资源，请先使用CPU运行时进行编码和测试非GPU函数。在测试CUDA或GPU加速代码时再切换到GPU运行时。这种方法能确保GPU的高效使用，并防止在关键任务期间耗尽资源。

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsys10714/hw3.git
%cd /content/drive/MyDrive/10714/hw3

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

In [ ]:
# REQUIRED FOR MUGRADE
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW3_NAME = "hw3"

In [ ]:
!make

The make command reads the Makefile in the current directory. The Makefile contains rules that define how to build targets (like executables or libraries). For each target specified in the Makefile, make checks the timestamps of the target file and its dependencies (like .c, .cpp, or .h files). If any dependency has been modified recently, it must rebuild the target.

---

make命令会读取当前目录下的Makefile文件。该文件包含定义如何构建目标（如可执行文件或库文件）的规则。对于Makefile中指定的每个目标，make会检查目标文件及其依赖项（如.c、.cpp或.h文件）的时间戳。若任何依赖项近期被修改过，则必须重新构建该目标。

In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [ ]:
import sys
sys.path.append('./python')

## Getting familiar with the NDArray class

As you get started with this homework, you should first familiarize yourself with the `NDArray.py` class we have provided as a starting point for the assignment.  The code is fairly brief (it's ~500 lines, but a lot of these are comments provided for the functions you'll implement).

At its core, the NDArray class is a Python wrapper for handling operations on generic n-dimensional arrays.  Recall that virtually any such array will be stored internally as a vector of floating point values, i.e.,

```c++
float data[size];
```

and then the actual access to different dimensions of the array are all handled by additional fields (such as the array shape, strides, etc) that indicates how this "flat" array maps to n-dimensional structure.  In order to achieve any sort of reasonable speed, the "raw" operations (like adding, binary operations, but also more structured operations like matrix multiplication, etc), all need to be written at some level in some native language like C++ (including e.g., making CUDA calls).  But a large number of operations likes transposing, broadcasting, sub-setting of matrices, and other, can all be handled by just adjusting the high-level structure of the array, like it's strides.

The philosophy behind the NDArray class is that we want _all_ the logic for handling this structure of the array to be written in Python.  Only the "true" low level code that actually performs the raw underlying operations on the flat vector (as well as the code to manage these flat vectors, as they might need to e.g., be allocated on GPUs), is written in C++.  The precise nature of this separation will likely start to make more sense to you as you work through the assignment, but generally speaking everything that can be done in Python, is done in Python; often e.g., at the cost of some inefficiencies ... we call `.compact()` (which copies memory) liberally in order to make the underlying C++ implementations simpler.

In more detail, there are five fields within the NDArray class that you'll need to be familiar with (note that the real class member these all these fields is preceded by an underscore, e.g., `_handle`, `_strides`, etc, some of which are then exposed as a public property ... for all your code it's fine to use the internal, underscored version).

1. `device` - A object of type `BackendDevice`, which is a simple wrapper that contains a link to the underlying device backend (e.g., CPU or CUDA).
2. `handle` - A class objected that stores the underlying memory of the array.  This is allocated as a class of type `device.Array()`, though this allocation all happens in the provided code (specifically the `NDArray.make` function), and you don't need to worry about calling it yourself.
3. `shape` - A tuple specifying the size of each dimension in the array.
4. `strides` - A tuple specifying the strides of each dimension in the array.
5. `offset` - An integer indicating where in the underlying `device.Array` memory the array actually starts (it's convenient to store this so we can more easily manage pointing back to existing memory, without having to track allocations).

By manipulating these fields, even pure Python code can perform a lot of the needed operations on the array, such as permuting the dimensions (i.e., transposing), broadcasting, and more.  And then for the raw array manipulation calls, the `device` class has a number of methods (implemented in C++) that contains the necessary implementations.

There are a few points to note:

* Internally, the class can use _any_ efficient means of operating on arrays of data as a "device" backend.  Even, for example, a numpy array, but where instead of actually using the `numpy.ndarray` to represent the n-dimensional array, we just represent a "flat" 1D array in numpy, then call the relevant numpy methods to implement all the needed operators on this raw memory.  This is precisely what we do in the `ndarray_backend_numpy.py` file, which essentially provided a "stub reference" that just uses numpy for everything.  You can use this class to help you  better debug your own "real" implementations for the "native" CPU and GPU backends.
* Of particular importance for many of your Python implementations will be the `NDArray.make` call:
```python
def make(shape, strides=None, device=None, handle=None, offset=0):
```
which creates a new NDArray with the given shape, strides, device, handle, and offset.  If `handle` is not specified (i.e., no pre-existing memory is referenced), then the call will allocate the needed memory, but if handle _is_ specified then no new memory is allocated, but the new NDArray points the same memory as the old one.  It is important to efficient implementations that as many of your functions as possible _don't_ allocate new memory, so you will want to use this call in many cases to accomplish this.
* The NDArray has a `.numpy()` call that converts the array to numpy.  This is _not_ the same as the "numpy_device" backend: this creates an actual `numpy.ndarray` that is equivalent to the given NDArray, i.e., the same dimensions, shape, etc, though not necessarily the same strides (Pybind11 will reallocate memory for matrices that are returned in this manner, which can change the striding).

---

## 熟悉NDArray类

开始本作业时，你首先应该熟悉我们提供的`NDArray.py`类，这是本作业的起点代码。代码相当简洁（约500行，其中很多是你将实现函数的注释说明）。

NDArray类的核心是一个用于处理通用n维数组操作的Python包装器。请记住，实际上任何这样的数组在内部都将存储为浮点数值的向量，即：

```c++
float data[size];
```

然后对数组不同维度的实际访问都由额外字段（如数组形状、步长等）处理，这些字段指示了这个"扁平"数组如何映射到n维结构。为了达到合理的速度，"原始"操作（如加法、二元操作，以及更结构化的操作如矩阵乘法等）都需要在某种原生语言（如C++，包括调用CUDA）的某个层级上编写。但大量操作，如转置、广播、矩阵子集选择等，都可以仅通过调整数组的高层结构（如其步长）来处理。

NDArray类背后的理念是，我们希望处理数组结构的所有逻辑都用Python编写。只有实际对扁平向量执行原始底层操作的"真正"低级代码（以及管理这些扁平向量的代码，因为它们可能需要在GPU上分配等）是用C++编写的。随着你完成作业，这种分离的精确性质可能会变得更加清晰，但一般来说，所有能在Python中完成的操作都在Python中完成；通常，这可能会以一些低效为代价……我们频繁调用`.compact()`（这会复制内存）以使底层C++实现更简单。

更详细地说，你需要熟悉NDArray类中的五个字段（注意，这些字段的实际类成员都以下划线开头，例如`_handle`、`_strides`等，其中一些随后作为公共属性暴露……在你的所有代码中，使用带下划线的内部版本是可以的）：

1. `device` - 一个`BackendDevice`类型的对象，这是一个简单的包装器，包含指向底层设备后端（如CPU或CUDA）的链接。
2. `handle` - 一个存储数组底层内存的类对象。它被分配为`device.Array()`类型，尽管所有这些分配都发生在提供的代码中（特别是`NDArray.make`函数），你不需要担心自己调用它。
3. `shape` - 一个指定数组中每个维度大小的元组。
4. `strides` - 一个指定数组中每个维度步长的元组。
5. `offset` - 一个整数，指示数组实际在底层`device.Array`内存中的起始位置（存储这个很方便，这样我们可以更容易地管理指向现有内存，而无需跟踪分配）。

通过操作这些字段，即使是纯Python代码也可以执行许多所需的数组操作，如维度置换（即转置）、广播等。然后对于原始数组操作调用，`device`类有许多方法（在C++中实现），其中包含必要的实现。

有几点需要注意：

* 在内部，该类可以使用任何高效的数据数组操作方式作为"设备"后端。例如，甚至可以使用numpy数组，但我们不使用`numpy.ndarray`来表示n维数组，而只是在numpy中表示一个"扁平"的一维数组，然后调用相关的numpy方法来实现对该原始内存的所有所需操作。这正是我们在`ndarray_backend_numpy.py`文件中所做的，它本质上提供了一个"存根参考"，仅使用numpy处理一切。你可以使用这个类来帮助你更好地调试自己为"原生"CPU和GPU后端编写的"真实"实现。
* 对于你的许多Python实现，特别重要的是`NDArray.make`调用：
```python
def make(shape, strides=None, device=None, handle=None, offset=0):
```
该调用创建一个具有给定形状、步长、设备、句柄和偏移量的新NDArray。如果未指定`handle`（即未引用预先存在的内存），则调用将分配所需的内存，但如果指定了handle，则不分配新内存，新NDArray指向与旧数组相同的内存。对于高效实现来说，尽可能多的函数不分配新内存很重要，因此在许多情况下，你将希望使用此调用来实现这一点。
* NDArray有一个`.numpy()`调用，可将数组转换为numpy。这与"numpy_device"后端不同：这会创建一个与给定NDArray等效的实际`numpy.ndarray`，即具有相同的维度、形状等，尽管步长不一定相同（Pybind11将以这种方式返回的矩阵重新分配内存，这可能会改变步长）。

## Part 1: Python array operations

As a starting point for your class, implement the following functions in the `ndarray.py` file:

- `reshape()`
- `permute()`
- `broadcast_to()`
- `__getitem__()`

The inputs/outputs of these functions are all described in the docstring of the function stub.  It's important to emphasize that _none_ of these functions should reallocate memory, but should instead return NDArrays that share the same memory with `self`, and just use clever manipulation of shape/strides/etc in order to obtain the necessary transformations.

To get started you can refer to the hints mentioned in the class slides for transpose, broadcast and slicing operator.

One thing to note is that the `__getitem__()` call, unlike numpy, will never change the number of dimensions in the array.  So e.g., for a 2D NDArray `A`, `A[2,2]` would return a still-2D with one row and one column.  And e.g. `A[:4,2]` would return a 2D NDarray with 4 rows and 1 column.

You can rely on the `ndarray_backend_numpy.py` module for all the code in this section.  You can also look at the results of equivalent numpy operations (the test cases should illustrate what these are).

After implementing these functions, you should pass/submit the following tests.  Note that we test all of these four functions within the test below, and you can incrementally try to pass more tests as you implement each additional function.

---

## 第一部分：Python数组操作

作为你课程的起点，请在`ndarray.py`文件中实现以下函数：

- `reshape()`
- `permute()`
- `broadcast_to()`
- `__getitem__()`

这些函数的输入/输出都在函数存根的文档字符串中描述。需要强调的是，这些函数都不应该重新分配内存，而应该返回与`self`共享相同内存的NDArray，只需巧妙操作形状/步长等来获得必要的转换。

开始时，你可以参考课程幻灯片中提到的转置、广播和切片操作提示。

需要注意的一点是，与numpy不同，`__getitem__()`调用永远不会改变数组的维度数。因此，例如对于一个2D NDArray `A`，`A[2,2]`将返回一个仍然是2D的数组，但只有一行一列。而`A[:4,2]`将返回一个具有4行1列的2D NDArray。

在本节的所有代码中，你可以依赖`ndarray_backend_numpy.py`模块。你也可以查看等效numpy操作的结果（测试用例应该说明这些是什么）。

实现这些函数后，你应该通过/提交以下测试。请注意，我们在下面的测试中测试了所有这四个函数，你可以在实现每个附加函数时逐步尝试通过更多测试。

In [ ]:
!python3 -m pytest -v -k "(permute or reshape or broadcast or getitem) and cpu and not compact"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_python_ops"

## Part 2: CPU Backend - Compact and setitem

Implement the following functions in `ndarray_backend_cpu.cc`:
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

To see why these are all in the same category, let's consider the implementation of the `Compact()` function.  Recall that a matrix is considered compact if it is layed out sequentially in memory in "row-major" form (but really a generalization of row-many to higher dimensional arrays), i.e. with the last dimension first, followed by the second to last dimension, etc, all the way to the first.  In our implementation, we also require that the total size of allocated backend array be equal to the size of the array (i.e., the underlying array also can't have any data before or after the array data, which e.g., implies that the `.offset` field equals zero).

Now let's consider, using a 3D array as a an example, of how a compact call might work.  Here `shape` and `strides` are the shape and strides of the matrix being compacted (i.e., before we have compacted it).

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[cnt++] = in[strides[0]*i + strides[1]*j + strides[2]*k];
```
In other words, we're converting from a stride-based representation of the matrix to a purely sequential one.

Now, the challenge in implementing `Compact()` is that you want the method to work for any number of input dimensions.  It's easy to specialize for different fixed-dimension-size arrays, but for a generic implementation, you'll want to think about how to do this same operation where you effectively want a "variable number of for loops".  As a hint, one way to do this is to maintain a vector of indices (of size equal to the number of dimensions), and then manually increment them in a loop (including a "carry" operation when any of the reaches their maximum size).

However, if you get really stuck with this, you can alway use the fact that we're probably not going to ask you to deal with matrices of more than 6 dimensions (though we _will_ use 6 dimensions, for the im2col operation we discussed in class).


#### The connection to setitem
The setitem functionality, while seemingly quite different, is actually intimately related to `Compact()`.  `__setitem()__` is the Python function called when setting some elements of the object, i.e.,
```python
A[::2,4:5,9] = 0 # or = some_other_array
```
How would you go about implementing this?  In the `__getitem()__` call above, you already implemented a method to take a subset of a matrix without copying (but just modifying strides).  But how would you actually go about _setting_ elements of this array?  In virtually all the other settings in this homework, we call `.compact()` before setting items in an output array, but in this case it doesn't work: calling `.compact()` would copy the subset array to some new memory, but the whole point of the `__setitem__()` call is that we want to modify existing memory.

If you think about this for a while, you'll realize that the answer looks a lot like `.compact()` but in reverse.  If we want to assign a (itself already compact) right hand side matrix to a `__getitem()__` results, then we need to here like `shape` and `stride` be those fields of the _output_ matrix.  Then we could implement the setitem call as follows

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[strides[0]*i + strides[1]*j + strides[2]*k] = in[cnt++]; // or "= val;"
```
Due to this similarity, if you implement your indexing strategy in a modular fashion, you'll be able to reuse it between the `Compact()` call and the `EwiseSetitem()` and `ScalarSetitem()` calls.

**Note**: Don't forget to run make before executing the tests to rebuild your modified C++ code.

---

## 第二部分：CPU后端 - Compact和setitem

在`ndarray_backend_cpu.cc`中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

要理解为什么这些函数属于同一类别，让我们考虑`Compact()`函数的实现。回想一下，如果一个矩阵以"行主序"形式（实际上是行主序到高维数组的推广）在内存中顺序布局，即最后一个维度在前，然后是倒数第二个维度，依此类推，直到第一个维度，那么该矩阵被认为是紧凑的。在我们的实现中，我们还要求分配的后端数组的总大小等于数组的大小（即底层数组在数组数据之前或之后不能有任何数据，这意味着`.offset`字段等于零）。

现在让我们以3D数组为例，考虑compact调用可能如何工作。这里的`shape`和`strides`是要压缩的矩阵的形状和步长（即在我们压缩它之前）。

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[cnt++] = in[strides[0]*i + strides[1]*j + strides[2]*k];
```
换句话说，我们正在从基于步长的矩阵表示转换为纯顺序表示。

现在，实现`Compact()`的挑战在于你希望该方法适用于任意数量的输入维度。为不同的固定维度大小的数组专门实现很容易，但对于通用实现，你需要思考如何执行相同的操作，其中你实际上需要"可变数量的for循环"。作为提示，一种方法是维护一个索引向量（大小等于维度数），然后在循环中手动递增它们（包括当任何索引达到其最大值时的"进位"操作）。

然而，如果你真的卡住了，你可以利用我们可能不会要求你处理超过6维的矩阵这一事实（尽管我们*将*使用6维，用于我们在课堂上讨论的im2col操作）。

#### 与setitem的连接
setitem功能虽然看起来完全不同，但实际上与`Compact()`密切相关。`__setitem()__`是设置对象某些元素时调用的Python函数，例如：
```python
A[::2,4:5,9] = 0 # 或 = some_other_array
```
你将如何实现这个？在上面的`__getitem()__`调用中，你已经实现了一种在不复制的情况下获取矩阵子集的方法（只是修改步长）。但如何实际*设置*这个数组的元素？在本作业的几乎所有其他设置中，我们在设置输出数组中的项目之前调用`.compact()`，但在这里不起作用：调用`.compact()`会将子集数组复制到某个新内存，但`__setitem__()`调用的重点是我们想要修改现有内存。

如果你思考一会儿，你会意识到答案看起来很像`.compact()`，但是反向的。如果我们想将一个（本身已经紧凑的）右侧矩阵分配给`__getitem()__`的结果，那么我们需要让`shape`和`stride`成为*输出*矩阵的这些字段。然后我们可以如下实现setitem调用：

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[strides[0]*i + strides[1]*j + strides[2]*k] = in[cnt++]; // 或 "= val;"
```
由于这种相似性，如果你以模块化的方式实现索引策略，你将能够在`Compact()`调用与`EwiseSetitem()`和`ScalarSetitem()`调用之间重用它。

**注意**：在执行测试之前不要忘记运行make以重新构建你修改的C++代码。

In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_compact_setitem"

## Part 3: CPU Backend - Elementwise and scalar operations

Implement the following functions in `ndarray_backend_cpu.cc`:

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

You can look at the included
`EwiseAdd()` and `ScalarAdd()` functions (plus the invocations from `NDArray` in order to understand the required format of these functions.

Note that unlike the remaining functions mentioned here, we do not include function stubs for each of these functions.  This is because, while you can implement these naively just through implementing each function separately, though this will end up with a lot of duplicated code.  You're welcome to use e.g., C++ templates or macros to address this problem (but these would only be exposed internally, not to the external interface).

---

## 第三部分：CPU后端 - 逐元素和标量操作

在`ndarray_backend_cpu.cc`中实现以下函数：

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

你可以参考已包含的`EwiseAdd()`和`ScalarAdd()`函数（以及`NDArray`中的调用），以了解这些函数所需的格式。

请注意，与这里提到的其余函数不同，我们没有为每个这些函数提供函数存根。这是因为，虽然你可以通过单独实现每个函数来朴素地实现这些功能，但这最终会导致大量重复代码。你可以使用C++模板或宏来解决这个问题（但这些只会在内部暴露，不会暴露给外部接口）。

In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_ops"

## Part 4: CPU Backend - Reductions


Implement the following functions in `ndarray_backend_cpu.cc`:

* `ReduceMax()`
* `ReduceSum()`

In general, the reduction functions `.max()` and `.sum()` in NDArray take the max or sum across a specified axis specified by the `axis` argument (or across the entire array when `axis=None`); note that we don't support axis being a set of axes, though this wouldn't be too hard to add if you desired (but it's not in the interface you should implement for the homework).

Because summing over individual axes can be a bit tricky, even for compact arrays, these functions (in Python) in Python simplify things by permuting the last axis to the be the one reduced over (this is what the `reduce_view_out()` function in NDArray does), then compacting the array.  So for your `ReduceMax()` and `ReduceSum()` functions you implement in C++, you can assume that both the input and output arrays are contiguous in memory, and you want to just reduce over contiguous elements of size `reduce_size` as passed to the C++ functions. 

---

## 第四部分：CPU后端 - 归约操作

在`ndarray_backend_cpu.cc`中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

一般来说，NDArray中的归约函数`.max()`和`.sum()`在由`axis`参数指定的轴上取最大值或求和（当`axis=None`时在整个数组上）；请注意，我们不支持axis作为一组轴，尽管如果你想要添加这个功能并不太难（但这不是你应该为作业实现的接口的一部分）。

因为即使在紧凑数组上，对单个轴求和也可能有些棘手，这些Python函数通过将最后一个轴置换为要归约的轴来简化事情（这是NDArray中`reduce_view_out()`函数所做的），然后压缩数组。因此，对于你在C++中实现的`ReduceMax()`和`ReduceSum()`函数，你可以假设输入和输出数组在内存中都是连续的，并且你只想在传递给C++函数的`reduce_size`大小的连续元素上进行归约。

In [ ]:
!python3 -m pytest -v -k "reduce and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_reductions"

## Part 5: CPU Backend - Matrix multiplication

Implement the following functions in `ndarray_backend_cpu.cc`:

* `Matmul()`
* `MatmulTiled()`
* `AlignedDot()`

The first implementation, `Matmul()` can use the naive three-nested-for-loops algorithm for matrix multiplication.  However, the `MatmulTiled()` performs the same matrix multiplication on memory laid out in tiled form, i.e., as a contiguous 4D array
```c++
float[M/TILE][N/TILE][TILE][TILE];
```

Make sure to initialize the output matrix to zero before populating it with the correct values during matrix multiplication.

Note that the Python `__matmul__` code already does the conversion to tiled form when all sizes of the matrix multiplication are divisible by `TILE`, so your code just needs to implement the multiplication in this form.  In order to make the methods efficient, you will want to make use of (after you implement it), the `AlignedDot()` function, which will enable the compiler to efficiently make use of vector operations and proper caching.  The output matrix will also be in the tiled form above, and the Python backend will take care of the conversion to a normal 2D array.

Note that in order to get the most speedup possible from you tiled version, you may want to use the clang compiler with colab instead of gcc.  To do this, run the following command before building your code.

---

## 第五部分：CPU后端 - 矩阵乘法

在`ndarray_backend_cpu.cc`中实现以下函数：

* `Matmul()`
* `MatmulTiled()`
* `AlignedDot()`

第一个实现`Matmul()`可以使用朴素的三重嵌套for循环算法进行矩阵乘法。然而，`MatmulTiled()`在平铺形式的内存布局上执行相同的矩阵乘法，即作为一个连续的4D数组：
```c++
float[M/TILE][N/TILE][TILE][TILE];
```

在矩阵乘法期间用正确值填充输出矩阵之前，请确保将输出矩阵初始化为零。

请注意，当矩阵乘法的所有大小都能被`TILE`整除时，Python的`__matmul__`代码已经执行了到平铺形式的转换，因此你的代码只需要以这种形式实现乘法。为了使方法高效，你将需要使用（在你实现之后）`AlignedDot()`函数，这将使编译器能够有效地利用向量操作和适当的缓存。输出矩阵也将采用上述平铺形式，Python后端将负责转换为正常的2D数组。

请注意，为了从平铺版本获得最大的加速效果，你可能希望使用clang编译器而不是gcc。在构建代码之前运行以下命令。

In [ ]:
!python3 -m pytest -v -k "matmul and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_matmul"

In [ ]:
!export CXX=/usr/bin/clang++ && make

## Part 6: CUDA Backend - Compact and setitem

Implement the following functions in `ndarray_backend_cuda.cu`:
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`


For this portion, you'll implement the compact and setitem calls in the CUDA backend.  This is fairly similar to the C++ version, however, depending on how you implemented that function, there could also be some substantial differences.  We specifically want to highlight a few differences between the C++ and the CUDA implementations, however.

First, as with the example functions implemented in the CUDA backend code, for all the functions above you will actually want to implement two functions: the basic functions listed above that you will call from Python, and the corresponding CUDA kernels that will actually perform the computation.  For the most part, we only provide the prototype for the "base" function in the `ndarray_backend_cuda.cu` file, and you will need to define and implement the kernel function yourself.  However, to see how these work, for the `Compact()` call we are providing you with the _complete_ `Compact()` call, and the function prototype for the `CompactKernel()` call.

One thing you may notice is the seemingly odd use of a `CudaVec` struct, which is a struct used to pass shape/stride parameters.  In the C++ version we used the STL `std::vector` variables to store these inputs (and the same is done in the base `Compact()` call, but CUDA kernels cannot operation on STL vectors, so something else is needed).  Furthermore, although we _could_ convert the vectors to normal CUDA arrays, this would be rather cumbersome, as we would need to call `cudaMalloc()`, pass the parameters as integer pointers, then free them after the calls.  Of course such memory management is needed for the actual underlying data in the array, but it seems like overkill to do it for just passing a variable-sized small tuple of shape/stride values.  The solution is to create a struct that has a "maximize" size for the number of dimensions an array can have, and then just store the actual shape/stride data in the first entries of these fields.  This is all done by the included `CudaVec` struct and `VecToCuda()` function, and you can just use these as provided for all the CUDA kernels that require passing shape/strides to the kernel itself.

The other (more conceptual) big difference between the C++ and CUDA implementations of `Compact()` is that in C++ you will typically loop over the elements of the non-compact array sequentially, which allows you to perform some optimizations with respect to computing the corresponding indices between the compact and non-compact arrays.  In CUDA, you cannot do this, and will need to implement code that can directly map from an index in the compact array to one in the strided array.

As before, we recommend you implement your code in such as way that it can easily be re-used between the `Compact()`, and `Setitem()` calls.  As a short note, remember that if you want to call a (separate, non-kernel) function from kernel code, you need to define it as a `__device__` function. In CUDA, `__global__` is used to define a function that runs on the GPU and is called from the CPU host, while `__device__` defines a function that runs and is called only on the GPU from other GPU function code

---

## 第六部分：CUDA后端 - 压缩和设置项操作

在`ndarray_backend_cuda.cu`中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

在这一部分，你将在CUDA后端实现压缩和设置项调用。这与C++版本相当类似，但根据你实现该函数的方式，也可能存在一些显著差异。不过，我们特别想强调C++和CUDA实现之间的几个区别。

首先，与CUDA后端代码中实现的示例函数一样，对于上述所有函数，你实际上需要实现两个函数：从Python调用的上述基本函数，以及实际执行计算的相应CUDA内核。在大多数情况下，我们只在`ndarray_backend_cuda.cu`文件中提供"基础"函数的原型，你需要自己定义和实现内核函数。然而，为了了解这些函数的工作原理，对于`Compact()`调用，我们为你提供了_完整_的`Compact()`调用和`CompactKernel()`调用的函数原型。

你可能会注意到一个看似奇怪的`CudaVec`结构体的使用，这是一个用于传递形状/步长参数的结构体。在C++版本中，我们使用STL的`std::vector`变量来存储这些输入（基础`Compact()`调用也是如此），但CUDA内核无法操作STL向量，因此需要其他解决方案。此外，虽然我们_可以_将向量转换为普通的CUDA数组，但这将相当繁琐，因为我们需要调用`cudaMalloc()`，将参数作为整数指针传递，然后在调用后释放它们。当然，对于数组中实际的基础数据需要这样的内存管理，但仅仅为了传递可变大小的小型形状/步长值元组而这样做似乎有些过度。解决方案是创建一个结构体，该结构体具有数组可以拥有的维度数量的"最大"大小，然后将实际的形状/步长数据存储在这些字段的前几个条目中。所有这些都由包含的`CudaVec`结构体和`VecToCuda()`函数完成，你可以在所有需要将形状/步长传递给内核本身的CUDA内核中使用这些提供的功能。

`Compact()`的C++和CUDA实现之间的另一个（更概念性的）大区别是，在C++中，你通常会顺序遍历非紧凑数组的元素，这允许你在计算紧凑数组和非紧凑数组之间的对应索引方面进行一些优化。在CUDA中，你无法这样做，需要实现能够直接从紧凑数组中的索引映射到跨步数组中的索引的代码。

和之前一样，我们建议你以这样的方式实现代码，使其可以在`Compact()`和`Setitem()`调用之间轻松重用。简要说明一下，请记住，如果你想从内核代码调用一个（单独的、非内核的）函数，需要将其定义为`__device__`函数。在CUDA中，`__global__`用于定义在GPU上运行并从CPU主机调用的函数，而`__device__`定义仅在GPU上运行并从其他GPU函数代码调用的函数。

In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_compact_setitem"

## Part 7: CUDA Backend - Elementwise and scalar operations

Implement the following functions in `ndarray_backend_cuda.cu`:

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

Again, we don't provide these function prototypes, and you're welcome to use C++ templates or macros to make this implementation more compact.  You will also want to uncomment the appropriate regions of the Pybind11 code once you've implemented each function.

---

## 第七部分：CUDA后端 - 逐元素和标量操作

在`ndarray_backend_cuda.cu`中实现以下函数：

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

同样，我们不提供这些函数原型，你可以使用C++模板或宏来使实现更紧凑。在实现每个函数后，你还需要取消注释Pybind11代码的相应区域。

In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_ops"

## Part 8: CUDA Backend - Reductions


Implement the following functions in `ndarray_backend_cuda.cu`:

* `ReduceMax()`
* `ReduceSum()`

You can take a fairly simplistic approach here, and just use a separate CUDA thread for each individual reduction item: i.e., if there is a 100 x 20 array you are reducing over the second dimension, you could have 100 threads, each of which individually processed its own 20-dimensional array..  This is particularly inefficient for the `.max(axis=None)` calls, but we won't worry about this for the time being.  If you want a more industrial-grade implementation, you use a hierarchical mechanism that first aggregated across some smaller span, then had a secondary function that aggregated across _these_ reduced arrays, etc.  But this is not needed to pass the tests.

---

## 第八部分：CUDA后端 - 归约操作

在`ndarray_backend_cuda.cu`中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

你可以采用相当简单的方法，只需为每个独立的归约项使用单独的CUDA线程：例如，如果有一个100 x 20的数组需要在第二维度上进行归约，你可以使用100个线程，每个线程独立处理自己的20维数组。这对于`.max(axis=None)`调用来说特别低效，但我们暂时不会担心这个问题。如果你想要一个更工业级的实现，可以使用分层机制，首先在一些较小的跨度上进行聚合，然后使用辅助函数在这些_已归约的_数组上进行聚合，等等。但通过测试并不需要这样做。

In [ ]:
!python3 -m pytest -v -k "reduce and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_reductions"

## Part 9: CUDA Backend - Matrix multiplication

Implement the following functions in `ndarray_backend_cuda.cu`:

* `Matmul()`

Finally, as your final exercise, you'll implement matrix multiplication on the GPU.  Your implementation here can roughly follow the presentation in class.  While you can pass the tests using fairly naive code here (i.e., you could just have a separate thread for each (i,j) location in the matrix, doing the matrix multiplication efficiently (to make it actually faster than a CPU version) requires cooperative fetching and the block shared memory register tiling covered in class.  Try to implement using these methods, and see how much faster you can get your code than the C++ (or numpy) backends.

---

## 第九部分：CUDA后端 - 矩阵乘法

在`ndarray_backend_cuda.cu`中实现以下函数：

* `Matmul()`

最后，作为你的最终练习，你将在GPU上实现矩阵乘法。你的实现可以大致遵循课堂上的讲解。虽然你可以使用相当朴素的方法通过测试（即，你可以为矩阵中的每个(i,j)位置使用单独的线程），但要高效地进行矩阵乘法（使其实际比CPU版本更快）需要协作获取和课堂上介绍的块共享内存寄存器平铺技术。尝试使用这些方法实现，看看你的代码能比C++（或numpy）后端快多少。

In [ ]:
!python3 -m pytest -v -k "matmul and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_matmul"